# G1 SIT — train on free Colab GPU → run on your Mac

**ワンクリックで回す手順:** メニュー **Runtime → Run all**（実行 → すべてのセルを実行）。

前提: **Runtime → Change runtime type → T4 GPU** を選んでおくこと（無料）。

やること: GPU確認 → Playground導入 → 学習スクリプト取得(GitHub) → SIT学習 → 結果DL。
Driveは使いません(マウント不要)。学習済みポリシーは最後にMacへDLします。

In [ ]:
# 1) GPUが有効か確認(T4が見えればOK)
import subprocess
r = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True)
if r.returncode == 0 and r.stdout.strip():
    print(r.stdout.strip())
    print("✅ GPU OK — 次のセルへ")
else:
    print("⚠️ GPUなし → メニュー [ランタイム → ランタイムのタイプを変更 → T4 GPU] を選び、")
    print("   接続し直してからこのセルを再実行してください。")
    print("   それでも付かない場合は無料枠の上限の可能性 → 時間をおく / Kaggle / 少ステップで再試行。")

In [ ]:
# 2) MuJoCo Playground + JAX(CUDA) を導入
!pip install -q playground "jax[cuda12]"
import jax; print('JAX devices:', jax.devices())   # -> [CudaDevice(id=0)] が出れば成功

In [ ]:
# 3) 学習スクリプトをGitHubから取得(常に最新・Drive不要)
#    g1_train_colab.py は内部で g1_sit_env.py を import するので両方ファイルで置く
!wget -q -O g1_train_colab.py https://raw.githubusercontent.com/fumito072/MuJoCo-skills/main/training/g1_train_colab.py
!wget -q -O g1_sit_env.py     https://raw.githubusercontent.com/fumito072/MuJoCo-skills/main/training/g1_sit_env.py
!ls -la g1_*.py

In [ ]:
# 4) SIT タスクを学習(eval_reward が表示されれば成功。伸びを見て報酬を調整していく)
!python g1_train_colab.py --task sit --steps 40_000_000 --out /content/g1_sit

In [ ]:
# 5) 学習済みポリシーをMacへダウンロード(models/policies/ に置く)
from google.colab import files
files.download('/content/g1_sit_params.pkl')
files.download('/content/g1_sit_config.json')